# Seed Database

Loads `finance.csv` into two Supabase tables:
1. **`stocks`** — the reference table (ticker, company, sector, etc.)
2. **`fundamentals`** — the quarterly financial snapshot

`stocks` must be loaded first because `fundamentals` has a foreign key pointing to it.

We use **upsert** instead of insert so this notebook is safe to re-run — it updates existing rows instead of erroring on duplicates.

In [16]:
# Standard imports
# pandas — reads the CSV and lets us slice/rename columns before loading
# supabase — Python client to talk to our database
# dotenv — loads SUPABASE_URL and SUPABASE_KEY from the .env file
# datetime — used to stamp today's date on every fundamentals row

import pandas as pd
from supabase import create_client
from dotenv import load_dotenv
from datetime import date
import os

print("Libraries loaded.")

Libraries loaded.


In [17]:
# Connect to Supabase
# load_dotenv() reads the .env file in the project root and makes the values
# available via os.getenv() — keeps secrets out of the code

load_dotenv(dotenv_path="../.env")

url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_KEY")

supabase = create_client(url, key)
print("Connected successfully")

Connected successfully


In [18]:
# Load the CSV
# The CSV lives one folder up in dataset/
# dtype=str on ticker prevents pandas from mangling symbols like 'BTC-USD'

df = pd.read_csv("../dataset/finance.csv", dtype={"ticker": str})

print(f"Rows loaded: {len(df)}")
print(f"Columns: {len(df.columns)}")
df.head(3)

Rows loaded: 560
Columns: 76


,ticker,company,theme,asset_type,symbol,exchange,quoteType,currentPrice,previousClose,dayHigh,...,category,ytdReturn,threeYearAverageReturn,fiveYearAverageReturn,circulatingSupply,maxSupply,totalSupply,volume24Hr,sector,industry
0,MMM,3M,S&P 500,Equity,MMM,NYQ,EQUITY,152.85,155.17,154.2899,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Industrials,Conglomerates
1,AOS,A. O. Smith,S&P 500,Equity,AOS,NYQ,EQUITY,57.62,58.01,57.7700,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Industrials,Specialty Industrial Machinery
2,ABT,Abbott Laboratories,S&P 500,Equity,ABT,NYQ,EQUITY,86.30,85.68,86.3950,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Healthcare,Medical Devices


---
## Part 1 — Load `stocks`

The `stocks` table only needs the 7 reference columns. We pull those out of the full CSV, drop any rows where ticker is missing (would violate the primary key), and deduplicate — some tickers appear in both the S&P 500 list and the personal watchlist.

In [19]:
# Pull only the 7 columns the stocks table expects
# Rename exchange column — CSV calls it 'exchange', table matches, but
# quoteType maps to asset_type only when asset_type is missing (ETFs/Crypto
# from the watchlist already have asset_type set)

stocks_df = df[[
    "ticker", "company", "sector", "industry",
    "theme", "asset_type", "exchange"
]].copy()

# Drop rows with no ticker — can't insert without a primary key
stocks_df = stocks_df.dropna(subset=["ticker"])

# Deduplicate — keep the first occurrence if a ticker shows up more than once
# (happens when a watchlist stock is also in the S&P 500)
stocks_df = stocks_df.drop_duplicates(subset=["ticker"], keep="first")

# Replace NaN with None so the JSON serializer sends null instead of 'NaN'
# Supabase (PostgreSQL) understands null — it doesn't understand the string NaN
stocks_df = stocks_df.astype(object).where(pd.notnull(stocks_df), None)

print(f"Stocks to load: {len(stocks_df)}")
stocks_df.head()

Stocks to load: 560


,ticker,company,sector,industry,theme,asset_type,exchange
0,MMM,3M,Industrials,Conglomerates,S&P 500,Equity,NYQ
1,AOS,A. O. Smith,Industrials,Specialty Industrial Machinery,S&P 500,Equity,NYQ
2,ABT,Abbott Laboratories,Healthcare,Medical Devices,S&P 500,Equity,NYQ
3,ABBV,AbbVie,Healthcare,Drug Manufacturers - General,S&P 500,Equity,NYQ
4,ACN,Accenture,Technology,Information Technology Services,S&P 500,Equity,NYQ


In [20]:
# Convert to a list of dicts — that's the format Supabase expects
# upsert = insert if new, update if the ticker already exists
# This makes the notebook safe to re-run without throwing duplicate key errors

stocks_records = stocks_df.to_dict(orient="records")

response = supabase.table("stocks").upsert(stocks_records).execute()

print(f"Upserted {len(response.data)} rows into stocks")

Upserted 560 rows into stocks


In [21]:
# Verify — pull a count and spot-check a few rows

result = supabase.table("stocks").select("*", count="exact").execute()
print(f"Total rows in stocks table: {result.count}")

# Show the first 5 rows as a sanity check
pd.DataFrame(result.data).head()

Total rows in stocks table: 560


,ticker,company,sector,industry,theme,asset_type,exchange
0,MMM,3M,Industrials,Conglomerates,S&P 500,Equity,NYQ
1,DDOG,Datadog,Technology,Software - Application,S&P 500,Equity,NMS
2,NUE,Nucor,Basic Materials,Steel,S&P 500,Equity,NYQ
3,DVA,DaVita,Healthcare,Medical Care Facilities,S&P 500,Equity,NYQ
4,ZBH,Zimmer Biomet,Healthcare,Medical Devices,S&P 500,Equity,NYQ


---
## Part 2 — Load `fundamentals`

The `fundamentals` table stores quarterly snapshots. The `finance.csv` is a point-in-time pull, so every row gets stamped with:
- `period = '2026-Q2'` — the quarter this data represents
- `snapshot_date = today` — the exact date it was pulled

Next time we re-pull (e.g. in Q3), we load with `period = '2026-Q3'` and those become new rows — that's how historical comparisons build up over time.

Many columns will be `null` for ETFs and Crypto (they don't have PE ratios, revenue, etc.) — that's expected and fine.

In [ ]:
# Map CSV column names to our table column names
# CSV uses camelCase (yfinance convention), table uses snake_case

COLUMN_MAP = {
    "totalRevenue":        "revenue",
    "ebitda":              "ebitda",
    "ebitdaMargins":       "ebitda_margin",
    "grossMargins":        "gross_margin",
    "operatingMargins":    "operating_margin",
    "profitMargins":       "profit_margin",
    "freeCashflow":        "free_cashflow",
    "operatingCashflow":   "operating_cashflow",
    "trailingPE":          "trailing_pe",
    "forwardPE":           "forward_pe",
    "pegRatio":            "peg_ratio",
    "priceToBook":         "price_to_book",
    "enterpriseValue":     "enterprise_value",
    "revenueGrowth":       "revenue_growth",
    "earningsGrowth":      "earnings_growth",
    "returnOnEquity":      "return_on_equity",
    "returnOnAssets":      "return_on_assets",
    "debtToEquity":        "debt_to_equity",
    "trailingEps":         "trailing_eps",
    "forwardEps":          "forward_eps",
}

# Pull ticker + all fundamental columns from the CSV
fund_df = df[["ticker"] + list(COLUMN_MAP.keys())].copy()

# Rename to match table columns
fund_df = fund_df.rename(columns=COLUMN_MAP)

# Stamp every row with the quarter and pull date
fund_df["period"]        = "2026-Q2"
fund_df["snapshot_date"] = date.today().isoformat()

# Drop rows with no ticker
fund_df = fund_df.dropna(subset=["ticker"])

# Replace NaN with None — astype(object) must come first so pandas doesn't
# silently revert None back to NaN in float64 columns
fund_df = fund_df.astype(object).where(pd.notnull(fund_df), None)

print(f"Fundamentals rows to load: {len(fund_df)}")
fund_df.head(3)

In [ ]:
# Upsert into fundamentals
# on_conflict tells Supabase: if a row with the same (ticker, period) already
# exists, update it — otherwise insert a new row

import math
import numpy as np

fund_records = fund_df.to_dict(orient="records")

# Belt-and-suspenders: catch any numpy.float64('nan') that survived the
# DataFrame-level replacement — json.dumps rejects NaN with allow_nan=False
fund_records = [
    {k: None if isinstance(v, (float, np.floating)) and not math.isfinite(v) else v
     for k, v in row.items()}
    for row in fund_records
]

response = supabase.table("fundamentals").upsert(
    fund_records,
    on_conflict="ticker,period"
).execute()

print(f"Upserted {len(response.data)} rows into fundamentals")

In [ ]:
# Verify — count rows and spot-check a familiar ticker

result = supabase.table("fundamentals").select("*", count="exact").execute()
print(f"Total rows in fundamentals table: {result.count}")

# Pull one row for a stock we know has full data (AAPL or AMD)
check = supabase.table("fundamentals").select("*").eq("ticker", "AMD").execute()
pd.DataFrame(check.data)